## University Research & Innovation Dashboard
#### Description: 
This dashboard is organized into three primary analytical components: a **Heat Map** section, a **Keyword Search** section, and a **University Output Over Time** section. Each component includes configurable filters that allow users to refine the display of papers and patents based on selected criteria.

The **Heat Map** section provides three distinct visualizations: a papers heat map, a patents heat map, and a combined map integrating both data sources. Users can hover over any university to view the total number of papers and/or patents associated with that institution. When a university is selected, a detailed results table appears beneath the map, displaying papers and patents that match the currently applied filters for the chosen university.

The **Keyword Search** section includes a detailed results table that surfaces individual papers and patents. Users can apply multiple filters, including a keyword query, to narrow the dataset and focus on specific research topics, institutions, or categories.

The **University Output Over Time** section presents a series of line charts, one for each university, illustrating annual output across multiple research categories. This view enables users to identify trends, compare category‑level activity, and observe how each institution’s research focus evolves over time.

# Load Data 
The Load Data section prepares all datasets required for the dashboard by importing, cleaning, standardizing, and enriching multiple sources of paper, patent, and university information. It performs the following major functions:

**Reads all input files:** 
It loads paper counts, top‑5 paper titles, patent counts by year, top‑5 patent details, and university coordinate/Carnegie data from CSV files.

**Parses stringified lists:**  
The top‑5 papers and top‑5 patents files contain lists stored as strings. These are converted into proper Python lists of dictionaries so each work’s metadata (title, year, ID, citations, etc.) can be accessed programmatically.

**Normalizes university names:** 
A robust normalization function standardizes university names across all datasets by lowercasing, removing invisible characters, normalizing Unicode, fixing punctuation, and collapsing whitespace. This ensures consistent merging and accurate cross‑dataset matching.

**Cleans and standardizes coordinate data:**  
The coordinates file is corrected for inconsistent column naming (e.g., LATITUDE vs. latitude) so all datasets use uniform Latitude, Longitude, State, and Carnegie fields.

**Merges geographic and Carnegie information:**  
Latitude, longitude, state, and Carnegie classification are merged into both the paper and patent datasets using normalized university names, enriching each record with mapping and filtering attributes.

**Resolves duplicate merge columns:**  
Any redundant columns created during merging (e.g., Carnegie_x/Carnegie_y or State_x/State_y) are cleaned up and consolidated into a single canonical column.

**Removes entries missing coordinates:**  
Rows without latitude or longitude are dropped to ensure that all records can be plotted on maps without errors.

**Builds lookup dictionaries for top‑5 works:**  
Two dictionaries—one for papers and one for patents—are constructed.
Each dictionary is keyed by (University, Category) and stores the top‑5 works along with metadata such as titles, years, citation counts, patent IDs, and seed (subcategory) values.

**Extracts year lists:**
Paper and patent year lists are derived from their respective datasets, and a unified year list is created for global year‑slider filtering.

**Builds category lists:**  
Unique paper and patent categories are extracted for use in dropdown filters across the dashboard.

**Builds the Carnegie classification list:**  
A sorted list of all Carnegie types is created from the coordinates file for Carnegie‑based filtering.

**Defines helper utilities:**  
Utility functions are included for stripping HTML tags and extracting patent IDs from top‑patent lists.

**Defines U.S. region groupings:**  
A dictionary maps each region (Northeast, Midwest, South, West) to its corresponding states, enabling region‑based filtering in the dashboard.

In [1]:
import pandas as pd
from dash import Dash, html, dcc, dash_table, Input, Output, State, callback, callback_context
import plotly.express as px
import json
import unicodedata
import re
import ast

In [2]:
# ---------------------------------------------------------
# READ FILES
# ---------------------------------------------------------

# Paper Files (UPDATED)
PaperYears = pd.read_csv("pub_counts_by_year_category_seed_institution.csv")
Top5Papers = pd.read_csv("tiered_openalex_top5_titles.csv")

# Patents Files
PatentYears = pd.read_csv("patent_counts_by_year_category_seed_institution.csv")
Top5Patents = pd.read_csv("tiered_patent_top5_full.csv")

# University Coordinates File
locations = pd.read_csv("r1_r2_coordinates.csv", encoding="latin1")

# ---------------------------------------------------------
# PARSE STRINGIFIED LISTS IN TOP‑5 FILES
# ---------------------------------------------------------

def parse_list_column(df, colname):
    df[colname] = df[colname].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else (x if isinstance(x, list) else [])
    )
    return df

Top5Papers = parse_list_column(Top5Papers, "top_works")
Top5Patents = parse_list_column(Top5Patents, "top_patents")

# ---------------------------------------------------------
# STRONG UNIVERSITY NAME NORMALIZATION
# ---------------------------------------------------------

def normalize_univ(x):
    if pd.isna(x):
        return ""
    x = str(x)

    x = x.lower()
    x = unicodedata.normalize("NFKC", x)
    x = x.replace("\u200b", "").replace("\xa0", " ")
    x = x.replace("–", "-").replace("—", "-")
    x = x.replace("`", "'").replace("’", "'").replace("‘", "'")
    x = x.replace("“", '"').replace("”", '"')
    x = re.sub(r"\s+", " ", x)

    return x.strip()

for df in [PaperYears, PatentYears, Top5Papers, Top5Patents, locations]:
    df["University"] = df["University"].apply(normalize_univ)

# ---------------------------------------------------------
# FIX COORDINATE COLUMN NAMES
# ---------------------------------------------------------

locations = locations.rename(columns={
    "LATITUDE": "Latitude",
    "LONGITUDE": "Longitude",
    "latitude": "Latitude",
    "longitude": "Longitude"
})

# ---------------------------------------------------------
# MERGE COORDINATES INTO PaperYears + PatentYears
# ---------------------------------------------------------

PaperYears = PaperYears.merge(
    locations[["University", "Carnegie", "Latitude", "Longitude", "State"]],
    on="University",
    how="left",
    validate="many_to_one"
)

PatentYears = PatentYears.merge(
    locations[["University", "Carnegie", "Latitude", "Longitude", "State"]],
    on="University",
    how="left",
    validate="many_to_one"
)

# ---------------------------------------------------------
# FIX DUPLICATE MERGE COLUMNS
# ---------------------------------------------------------

def fix_merge_columns(df):
    if "Carnegie_y" in df.columns:
        df = df.drop(columns=["Carnegie_x"], errors="ignore")
        df = df.rename(columns={"Carnegie_y": "Carnegie"})
    if "State_y" in df.columns:
        df = df.drop(columns=["State_x"], errors="ignore")
        df = df.rename(columns={"State_y": "State"})
    return df

PaperYears = fix_merge_columns(PaperYears)
PatentYears = fix_merge_columns(PatentYears)

# ---------------------------------------------------------
# DROP ROWS MISSING COORDINATES
# ---------------------------------------------------------

PaperYears = PaperYears.dropna(subset=["Latitude", "Longitude"])
PatentYears = PatentYears.dropna(subset=["Latitude", "Longitude"])

# ---------------------------------------------------------
# BUILD TOP‑5 LOOKUP DICTIONARIES (WITH SEED)
# ---------------------------------------------------------

Top5PapersDict = {}
for _, entry in Top5Papers.iterrows():
    key = (entry["University"], entry["Category"])
    Top5PapersDict.setdefault(key, [])
    for w in entry["top_works"]:
        Top5PapersDict[key].append({
            "title": w.get("title", ""),
            "year": w.get("year", ""),
            "openalex_url": w.get("openalex_url", ""),
            "cited_by_count": w.get("cited_by_count", ""),
            "seed": entry.get("Seed", "")
        })

Top5PatentsDict = {}
for _, entry in Top5Patents.iterrows():
    key = (entry["University"], entry["Category"])
    Top5PatentsDict.setdefault(key, [])
    for p in entry["top_patents"]:
        Top5PatentsDict[key].append({
            "patent_id": p.get("patent_id", ""),
            "patent_title": p.get("patent_title", ""),
            "patent_date": p.get("patent_date", ""),
            "organization": p.get("organization", ""),
            "seed": entry.get("Seed", "")
        })

# ---------------------------------------------------------
# BUILD PAPER YEAR LIST (FROM PaperYears)
# ---------------------------------------------------------

paper_years = sorted(PaperYears["year"].dropna().unique())

# ---------------------------------------------------------
# BUILD PATENT YEAR LIST (FROM PatentYears)
# ---------------------------------------------------------

patent_years = sorted(PatentYears["year"].dropna().unique())

# ---------------------------------------------------------
# CATEGORY LISTS
# ---------------------------------------------------------

paper_categories = sorted(PaperYears["Category"].dropna().unique())
patent_categories = sorted(PatentYears["Category"].dropna().unique())

# ---------------------------------------------------------
# CARNEGIE LIST
# ---------------------------------------------------------

all_carnegie = sorted(locations["Carnegie"].dropna().unique())

# ---------------------------------------------------------
# HELPER FUNCTION
# ---------------------------------------------------------

def strip_html(text):
    return re.sub(r"<.*?>", "", text)

# ---------------------------------------------------------
# Helper: Extract patent_id from top_patents list
# ---------------------------------------------------------

def extract_patent_id(top_patents):
    if isinstance(top_patents, list) and len(top_patents) > 0:
        first = top_patents[0]
        if isinstance(first, dict):
            return first.get("patent_id", "")
    return ""

# ---------------------------------------------------------
# REGIONS
# ---------------------------------------------------------

regions = {
    "All Regions": locations["State"].dropna().unique().tolist(),
    "Northeast": ["Maine","New Hampshire","Vermont","Massachusetts","Rhode Island","Connecticut","New York","New Jersey","Pennsylvania"],
    "Midwest": ["Ohio","Michigan","Indiana","Illinois","Wisconsin","Minnesota","Iowa","Missouri","North Dakota","South Dakota","Nebraska","Kansas"],
    "South": ["Delaware","Maryland","Virginia","West Virginia","Kentucky","North Carolina","South Carolina","Tennessee","Georgia","Florida","Alabama","Mississippi","Arkansas","Louisiana","Texas","Oklahoma"],
    "West": ["Washington","Oregon","California","Nevada","Idaho","Montana","Wyoming","Utah","Colorado","Arizona","New Mexico","Alaska","Hawaii"]
}

# ---------------------------------------------------------
# UNIFIED YEAR LIST FOR GLOBAL SLIDER
# ---------------------------------------------------------

all_years = sorted(
    set(PaperYears["year"].unique()).union(
        set(PatentYears["year"].unique())
    )
)


## Map Generators
The Map Generators section creates all three geographic visualizations used in the dashboard by aggregating university‑level output, generating detailed hover text, and rendering rainbow‑scaled heatmaps overlaid with black marker dots. It performs the following major functions:

**Builds hover text for papers and patents:**  
Generates formatted hover content for each university, including Carnegie classification and category‑level totals. Separate functions handle papers and patents, ensuring that hover text reflects the correct output type and count.

**Aggregates university‑level counts:**  
Groups paper and patent data by university and geographic coordinates, summing total output so each institution appears once on the map with accurate counts for the selected filters.

**Generates the papers heatmap:**  
Creates a rainbow‑colored density map based on total paper output and overlays black marker dots at each university location. Paper‑specific hover text is attached to each marker.

**Generates the patents heatmap:**  
Builds a parallel rainbow density map for patent output, again overlaying black dots and using patent‑specific hover text to display category totals and Carnegie information.

**Generates the combined heatmap:**  
Merges paper and patent aggregates, computes total output, constructs unified hover text showing papers, patents, and combined totals, and renders a rainbow heatmap with black markers for all institutions.

**Applies consistent map styling:**
All maps use the same basemap, rainbow heat scale, and black‑dot overlays to maintain visual consistency across the dashboard.

**Normalizes year values:**  
A helper function converts mixed year formats (integers, strings, timestamps) into clean integer years, ensuring that year‑based filtering works reliably across all map generators.

In [3]:
# ---------------------------------------------------------
# BUILD HOVER TEXT FOR PAPERS
# ---------------------------------------------------------
def build_paper_hover(univ, df):
    lines = []

    carnegie = df["Carnegie"].iloc[0]
    display_univ = univ.title()

    lines.append(f"<b>{display_univ}</b>")
    lines.append(f"Carnegie: {carnegie}")
    lines.append("")

    cat_totals = (
        df.groupby("Category")["works_count"]
          .sum()
          .reset_index()
    )

    for _, row in cat_totals.iterrows():
        lines.append(f"{row['Category']}: {int(row['works_count'])} papers")

    return "<br>".join(lines)


# ---------------------------------------------------------
# BUILD HOVER TEXT FOR PATENTS
# ---------------------------------------------------------
def build_patent_hover(univ, df):
    lines = []

    carnegie = df["Carnegie"].iloc[0]
    display_univ = univ.title()

    lines.append(f"<b>{display_univ}</b>")
    lines.append(f"Carnegie: {carnegie}")
    lines.append("")

    cat_totals = (
        df.groupby("Category")["patent_count"]
          .sum()
          .reset_index()
    )

    for _, row in cat_totals.iterrows():
        lines.append(f"{row['Category']}: {int(row['patent_count'])} patents")

    return "<br>".join(lines)


# ---------------------------------------------------------
# PAPERS MAP (RAINBOW HEATMAP + BLACK DOTS)
# ---------------------------------------------------------
def generate_papers_map(df):
    """
    df is now PaperYears filtered by:
    - category
    - year slider
    """

    if df.empty:
        return px.scatter_map(pd.DataFrame({"lat": [], "lon": []}), lat="lat", lon="lon")

    # Aggregate by university
    agg = (
        df.groupby(["University", "Latitude", "Longitude"])
          ["works_count"]
          .sum()
          .reset_index()
    )

    # Hover text
    hover_dict = {
        univ: build_paper_hover(univ, df[df["University"] == univ])
        for univ in agg["University"].unique()
    }

    agg["hover"] = agg["University"].map(hover_dict)

    # Rainbow heatmap
    fig = px.density_map(
        agg,
        lat="Latitude",
        lon="Longitude",
        z="works_count",
        zoom=3,
        color_continuous_scale="Rainbow"
    )

    # Black dots
    fig.add_scattermap(
        lat=agg["Latitude"],
        lon=agg["Longitude"],
        mode="markers",
        marker=dict(size=8, color="black"),
        text=agg["hover"]
    )

    fig.update_layout(map_style="carto-positron")
    return fig


# ---------------------------------------------------------
# PATENTS MAP (RAINBOW HEATMAP + BLACK DOTS)
# ---------------------------------------------------------
def generate_patents_map(df):
    """
    df is PatentYears filtered by:
    - category
    - year slider
    """

    if df.empty:
        return px.scatter_map(pd.DataFrame({"lat": [], "lon": []}), lat="lat", lon="lon")

    agg = (
        df.groupby(["University", "Latitude", "Longitude"])
          ["patent_count"]
          .sum()
          .reset_index()
    )

    hover_dict = {
        univ: build_patent_hover(univ, df[df["University"] == univ])
        for univ in agg["University"].unique()
    }

    agg["hover"] = agg["University"].map(hover_dict)

    fig = px.density_map(
        agg,
        lat="Latitude",
        lon="Longitude",
        z="patent_count",
        zoom=3,
        color_continuous_scale="Rainbow"
    )

    fig.add_scattermap(
        lat=agg["Latitude"],
        lon=agg["Longitude"],
        mode="markers",
        marker=dict(size=8, color="black"),
        text=agg["hover"]
    )

    fig.update_layout(map_style="carto-positron")
    return fig


# ---------------------------------------------------------
# COMBINED MAP (RAINBOW HEATMAP + BLACK DOTS)
# ---------------------------------------------------------
def generate_combined_map(df_papers, df_patents):
    """
    df_papers = PaperYears filtered by category + year
    df_patents = PatentYears filtered by category + year
    """

    # Aggregate papers
    agg_p = (
        df_papers.groupby(["University", "Latitude", "Longitude"])
                 ["works_count"]
                 .sum()
                 .reset_index()
    )

    # Aggregate patents
    agg_t = (
        df_patents.groupby(["University", "Latitude", "Longitude"])
                  ["patent_count"]
                  .sum()
                  .reset_index()
    )

    # Merge totals
    merged = pd.merge(
        agg_p,
        agg_t,
        on=["University", "Latitude", "Longitude"],
        how="outer"
    ).fillna(0)

    merged["total"] = merged["works_count"] + merged["patent_count"]

    # Hover text
    hover_dict = {}
    for univ in merged["University"].unique():
        p = int(merged.loc[merged["University"] == univ, "works_count"].iloc[0])
        t = int(merged.loc[merged["University"] == univ, "patent_count"].iloc[0])
        total = int(merged.loc[merged["University"] == univ, "total"].iloc[0])

        hover_dict[univ] = (
            f"<b>{univ.title()}</b><br>"
            f"Papers: {p}<br>"
            f"Patents: {t}<br>"
            f"Total: {total}"
        )

    merged["hover"] = merged["University"].map(hover_dict)

    # Rainbow heatmap
    fig = px.density_map(
        merged,
        lat="Latitude",
        lon="Longitude",
        z="total",
        zoom=3,
        color_continuous_scale="Rainbow"
    )

    # Black dots
    fig.add_scattermap(
        lat=merged["Latitude"],
        lon=merged["Longitude"],
        mode="markers",
        marker=dict(
            size=8,
            color="black",
            opacity=0.85
        ),
        text=merged["hover"]
    )

    fig.update_layout(map_style="carto-positron")
    return fig

# ---------------------------------------------------------
# HELPER — Normalize year values (int or timestamp string)
# ---------------------------------------------------------

def normalize_year(value):
    """
    Convert year values to integers.
    Handles:
    - int (returns as-is)
    - '2025-09-09T00:00:00'
    - '2023'
    - None
    """
    if value is None:
        return None

    # Already an int
    if isinstance(value, int):
        return value

    # Extract leading digits from strings
    if isinstance(value, str):
        digits = "".join(ch for ch in value if ch.isdigit())
        if len(digits) >= 4:
            return int(digits[:4])

    return None


# Layout
The Layout section defines the full user interface of the dashboard, organizing all maps, filters, tables, search tools, and time‑series components into a structured, interactive, tab‑based design. It performs the following major functions:

**Creates the main dashboard container:**  
Initializes the Dash application, sets the dashboard title, and constructs the top‑level layout that holds all tabs, maps, filters, tables, and search modules.

**Defines a three‑tab navigation system:**  
Implements tabs for the Papers Map, Patents Map, and Combined Map, allowing users to switch between analytical views while maintaining a unified interface structure.

**Builds the Papers Map tab:**  
Includes a category dropdown, a paper‑year range slider, the papers heatmap, and a dynamic table that displays top‑5 paper details for the selected university. The tab also includes headers, spacing, and a consistent table structure.

**Builds the Patents Map tab:**  
Provides a patent category dropdown, a patent‑year range slider, the patents heatmap, and a corresponding table that lists top‑5 patent records for the selected university. The tab is initially hidden and becomes visible when selected.

**Builds the Combined Map tab:**  
Integrates both paper and patent categories, includes a unified year slider, renders the combined heatmap, and displays a merged top‑5 table showing both paper and patent works for the selected university.

**Creates the unified search interface:**  
Adds a global search section where users can filter papers and patents across all universities using category filters, a combined year slider, content‑type selection, Carnegie classification, row limits, and keyword search.

**Provides category, content‑type, and Carnegie filters:**  
Allows users to refine search results by research category, choose whether to view papers, patents, or both, and restrict results to specific Carnegie classifications or all R1/R2 institutions.

**Adds keyword‑based search capabilities:**  
Includes a text search box that filters results by title, university name, category, subcategory, ID, state, and other metadata, enabling highly targeted queries across the entire dataset.

**Displays a comprehensive search results table:**  
Renders a unified table containing university level information (including papers, patents, titles, years, citations, and IDs) updated dynamically based on all applied filters.

**Builds the University Patent Output Over Time module:**  
Provides controls for selecting content type, choosing a geographic region, specifying the number of universities to display, and filtering institutions by name or state before generating time‑series graphs.

**Renders dynamic patent‑output graphs:**  
Displays line charts showing year‑by‑year paper and patent output for selected universities, enabling comparison across regions, institutions, and search criteria.

In [4]:
# ---------------------------------------------------------
# LAYOUT (MAPS + TABLES + SEARCH + PATENT OUTPUT OVER TIME)
# ---------------------------------------------------------

map = Dash(__name__)

map.layout = html.Div([

    html.H1("University Research & Innovation Dashboard"),

    # -----------------------------------------------------
    # TABS
    # -----------------------------------------------------
    dcc.Tabs(
        id="map-tabs",
        value="papers-tab",
        children=[
            dcc.Tab(label="Papers Map", value="papers-tab"),
            dcc.Tab(label="Patents Map", value="patents-tab"),
            dcc.Tab(label="Combined Map", value="combined-tab"),
        ]
    ),

    # -----------------------------------------------------
    # PAPERS TAB CONTENT
    # -----------------------------------------------------
    html.Div(id="papers-tab-content", children=[

        html.H3("Papers Map Filters"),

        dcc.Dropdown(
            id="papers-category-filter",
            options=[{"label": c, "value": c} for c in paper_categories],
            multi=True,
            placeholder="Select categories...",
            style={"marginTop": "10px", "width": "400px"},
        ),

        # PAPER-ONLY YEAR SLIDER
        html.Div([
            html.Label("Filter Papers by Year:", style={"fontWeight": "bold"}),
            dcc.RangeSlider(
                id="paper-year-slider",
                min=min(paper_years),
                max=max(paper_years),
                value=[min(paper_years), max(paper_years)],
                marks={int(y): str(int(y)) for y in paper_years},
                step=1,
                allowCross=False,
                pushable=1,
            )
        ], style={"width": "600px", "marginTop": "20px", "marginBottom": "20px"}),

        dcc.Graph(id="papers-map"),

        html.Hr(),

        html.H2(
            id="selected-university-title-papers",
            children="Select a university on the papers map"
        ),

        dash_table.DataTable(
            id="university-table-papers",
            columns=[
                {"name": "Title", "id": "title"},
                {"name": "Type", "id": "type"},
                {"name": "Category", "id": "category"},
                {"name": "Subcategory", "id": "Seed"},
                {"name": "Year", "id": "year"},
                {"name": "ID", "id": "work_id"},
            ],
            data=[],
            page_size=10,
            style_table={"overflowX": "auto"},
            style_cell={"textAlign": "left", "whiteSpace": "normal", "height": "auto"},
        ),
    ]),

    # -----------------------------------------------------
    # PATENTS TAB CONTENT
    # -----------------------------------------------------
    html.Div(id="patents-tab-content", style={"display": "none"}, children=[

        html.H3("Patents Map Filters"),

        dcc.Dropdown(
            id="patents-category-filter",
            options=[{"label": c, "value": c} for c in patent_categories],
            multi=True,
            placeholder="Select categories...",
            style={"marginTop": "10px", "width": "400px"},
        ),

        # PATENT-ONLY YEAR SLIDER
        html.Div([
            html.Label("Filter Patents by Year:", style={"fontWeight": "bold"}),
            dcc.RangeSlider(
                id="patent-year-slider",
                min=min(patent_years),
                max=max(patent_years),
                value=[min(patent_years), max(patent_years)],
                marks={int(y): str(int(y)) for y in patent_years},
                step=1,
                allowCross=False,
                pushable=1,
            )
        ], style={"width": "600px", "marginTop": "20px", "marginBottom": "20px"}),

        dcc.Graph(id="patents-map"),

        html.Hr(),

        html.H2(
            id="selected-university-title-patents",
            children="Select a university on the patents map"
        ),

        dash_table.DataTable(
            id="university-table-patents",
            columns=[
                {"name": "Title", "id": "title"},
                {"name": "Type", "id": "type"},
                {"name": "Category", "id": "category"},
                {"name": "Subcategory", "id": "Seed"},
                {"name": "Year", "id": "year"},
                {"name": "ID", "id": "work_id"},
            ],
            data=[],
            page_size=10,
            style_table={"overflowX": "auto"},
            style_cell={"textAlign": "left", "whiteSpace": "normal", "height": "auto"},
        ),
    ]),

    # -----------------------------------------------------
    # COMBINED TAB CONTENT
    # -----------------------------------------------------
    html.Div(id="combined-tab-content", style={"display": "none"}, children=[

        html.H3("Combined Papers + Patents Map Filters"),

        dcc.Dropdown(
            id="combined-category-filter",
            options=[{"label": c, "value": c}
                     for c in sorted(set(paper_categories + patent_categories))],
            multi=True,
            placeholder="Select categories...",
            style={"marginTop": "10px", "width": "400px"},
        ),

        # COMBINED YEAR SLIDER
        html.Div([
            html.Label("Filter Papers + Patents by Year:", style={"fontWeight": "bold"}),
            dcc.RangeSlider(
                id="combined-year-slider",
                min=min(all_years),
                max=max(all_years),
                value=[min(all_years), max(all_years)],
                marks={int(y): str(int(y)) for y in all_years},
                step=1,
                allowCross=False,
                pushable=1,
            )
        ], style={"width": "600px", "marginTop": "20px", "marginBottom": "20px"}),

        dcc.Graph(id="combined-map"),

        html.Hr(),

        html.H2(
            id="selected-university-title-combined",
            children="Select a university on the combined map"
        ),

        dash_table.DataTable(
            id="university-table-combined",
            columns=[
                {"name": "Title", "id": "title"},
                {"name": "Type", "id": "type"},
                {"name": "Category", "id": "category"},
                {"name": "Subcategory", "id": "Seed"},
                {"name": "Year", "id": "year"},
                {"name": "ID", "id": "work_id"},
            ],
            data=[],
            page_size=10,
            style_table={"overflowX": "auto"},
            style_cell={"textAlign": "left", "whiteSpace": "normal", "height": "auto"},
        ),
    ]),

    html.Hr(),

    # -----------------------------------------------------
    # SEARCH SECTION
    # -----------------------------------------------------
    html.H2("Search Papers and Patents"),

    dcc.Dropdown(
        id="search-category",
        options=[{"label": c, "value": c}
                 for c in sorted(set(paper_categories + patent_categories))],
        multi=True,
        placeholder="Filter by category...",
        style={"width": "600px", "marginBottom": "20px"}
    ),

    # SEARCH COMBINED YEAR SLIDER
    html.Div([
        html.Label("Filter Papers + Patents by Year:", style={"fontWeight": "bold"}),
        dcc.RangeSlider(
            id="search-year-slider",
            min=min(all_years),
            max=max(all_years),
            value=[min(all_years), max(all_years)],
            marks={int(y): str(int(y)) for y in all_years},
            step=1,
            allowCross=False,
            pushable=1,
        )
    ], style={"width": "600px", "marginBottom": "30px"}),

    html.Div([
        html.Div([
            html.Label("Content Type", style={"fontWeight": "bold"}),
            dcc.Dropdown(
                id="content",
                options=[
                    {"label": "Papers Only", "value": "papers"},
                    {"label": "Patents Only", "value": "patents"},
                    {"label": "Both", "value": "both"},
                ],
                value="both",
                clearable=False,
                style={"width": "180px"}
            ),
        ], style={"marginRight": "30px"}),

        html.Div([
            html.Label("Carnegie", style={"fontWeight": "bold"}),
            dcc.Dropdown(
                id="carnegie-filter",
                options=[{"label": "R1 and R2", "value": "All"}] +
                        [{"label": c, "value": c} for c in all_carnegie],
                value="All",
                clearable=False,
                style={"width": "180px"}
            ),
        ], style={"marginRight": "30px"}),

        html.Div([
            html.Label("Number of Rows", style={"fontWeight": "bold"}),
            dcc.Input(
                id="NUniversities",
                type="number",
                value=10,
                min=1,
                max=500,
                style={"width": "120px"}
            ),
        ]),
    ], style={"display": "flex", "flexWrap": "wrap", "marginBottom": "20px"}),

    html.Div([
        html.Label("Keyword Search", style={"fontWeight": "bold"}),
        dcc.Input(
            id="search-university",
            type="text",
            placeholder="Search title, university, category, ID, state...",
            style={"width": "500px", "marginTop": "5px"}
        ),
    ], style={"marginBottom": "25px"}),

    dash_table.DataTable(
        id="search-results-table",
        columns=[
            {"name": "University", "id": "University"},
            {"name": "State", "id": "State"},
            {"name": "Carnegie", "id": "Carnegie"},
            {"name": "Category", "id": "Category"},
            {"name": "Subcategory", "id": "Seed"},
            {"name": "Papers", "id": "works_count"},
            {"name": "Patents", "id": "us_patents_count"},
            {"name": "Type", "id": "type"},
            {"name": "Title", "id": "title"},
            {"name": "Year", "id": "year"},
            {"name": "Citations", "id": "cited_by_count"},
            {"name": "ID", "id": "ID"},
        ],
        data=[],
        page_size=20,
        style_table={"overflowX": "auto"},
        style_cell={"textAlign": "left", "whiteSpace": "normal", "height": "auto"},
    ),

    html.Hr(),

    # -----------------------------------------------------
    # UNIVERSITY OUTPUT OVER TIME (NEW)
    # -----------------------------------------------------
    html.H2("University Output Over Time"),

    html.Div([
        html.Label("Content Type", style={"fontWeight": "bold"}),
        dcc.Dropdown(
            id="uot-content-type",
            options=[
                {"label": "Papers Only", "value": "papers"},
                {"label": "Patents Only", "value": "patents"},
                {"label": "Both", "value": "both"},
            ],
            value="both",
            clearable=False,
            style={"width": "200px"}
        ),
    ], style={"marginBottom": "20px"}),

    html.Div(
        style={
            "display": "flex",
            "gap": "20px",
            "alignItems": "center",
            "marginTop": "15px",
            "marginBottom": "15px"
        },
        children=[

            dcc.Dropdown(
                id="patent-regionselect",
                options=[{"label": r, "value": r} for r in regions.keys()],
                value="All Regions",
                style={"width": "300px"}
            ),

            dcc.Input(
                id="patent-NUniversities",
                type="number",
                value=5,
                min=1,
                max=50,
                style={"width": "120px"}
            ),
        ],
    ),

    dcc.Input(
        id="patent-USearch",
        type="text",
        placeholder="Search university or state..."
    ),

    html.Div(id="patent-university-graphs"),

    html.Hr(),
])

# Callback
The Callbacks section controls all interactive behavior in the dashboard, updating maps, tables, search results, and time‑series graphs in response to user actions. It performs the following major functions:

**Handles tab switching logic:**  
Shows or hides the Papers, Patents, and Combined Map sections based on the user’s tab selection, ensuring that only the active tab’s content is visible.

**Updates the Papers Map dynamically:**  
Applies category and year range filters to the papers dataset, aggregates university level paper counts, generates hover text, and renders an updated rainbow heatmap with black marker dots whenever the user adjusts filters or the year slider.

**Updates the Patents Map dynamically:**  
Filters the patents dataset by category and year range, aggregates patent counts by university, builds patent‑specific hover text, and regenerates the patents heatmap with black markers in response to user input.

**Updates the Combined Map dynamically:**  
Applies category and year range filters to both papers and patents, merges the two datasets, computes total output, constructs unified hover text showing papers, patents, and combined totals, and renders an updated combined heatmap whenever filters change.

**Ensures consistent empty‑state behavior:**  
Returns an empty scatter map with correct styling when filters produce no matching data, maintaining a smooth and predictable user experience.

**Builds the Papers Map selected‑university table:**  
Extracts the clicked university from the papers map, normalizes the name, applies category and Carnegie filters, normalizes year values, retrieves the top‑5 papers for each matching category, and displays them in a structured table with titles, categories, subcategories, years, and IDs.

**Builds the Patents Map selected‑university table:**  
Identifies the selected university from the patents map, normalizes the name, applies category, year range, and Carnegie filters, normalizes year values, retrieves the top‑5 patents for each matching category, extracts subcategory information from multiple possible fields, and populates the table with real patent titles, subcategories, years, and patent IDs.

**Builds the Combined Map selected‑university table:**  
Normalizes year values for both papers and patents, applies category and year‑range filters, retrieves top‑5 papers and top‑5 patents for the selected university, and merges them into a unified table of works with consistent fields for titles, types, categories, subcategories, years, and IDs.

**Ensures accurate university identification:**  
Parses hover text from map clicks, strips HTML, normalizes university names, and uses these cleaned identifiers to match entries across all datasets.

**Applies Carnegie filtering consistently:**  
Checks whether the selected university matches the user’s Carnegie filter and returns an empty table when it does not.

**Builds the unified search results table:**  
Normalizes year values, applies year range, category, Carnegie, and content type filters, merges papers and patents on a unified year column, and constructs detailed rows for each top‑5 paper and patent associated with the filtered results.

**Supports keyword‑based filtering across multiple fields:**  
Filters the merged search results by matching the user’s keyword against university names, categories, subcategories, titles, states, and IDs, enabling highly targeted search queries.

**Sorts, deduplicates, and limits search output:**  
Removes duplicate works, sorts results by year in descending order, and returns only the top number of rows specified by the user.

**Builds the University Patent Output Over Time module:**  
Selects papers, patents, or both depending on user choice; applies region filters, keyword search, and ranking logic; aggregates year‑by‑year output for each university; and prepares the data for visualization.

**Generates dynamic patent‑output line graphs:**  
Creates separate line charts for each top‑ranked university, showing year‑by‑year paper, patent, or combined output across categories, and returns them as graph components for display.

In [5]:
# ---------------------------------------------------------
# CALLBACKS — MAPS + SEARCH (FULLY CORRECTED)
# ---------------------------------------------------------

# ---------------------------------------------------------
# TAB SWITCHING
# ---------------------------------------------------------

@map.callback(
    Output("papers-tab-content", "style"),
    Output("patents-tab-content", "style"),
    Output("combined-tab-content", "style"),
    Input("map-tabs", "value")
)
def toggle_tabs(tab):
    if tab == "papers-tab":
        return {"display": "block"}, {"display": "none"}, {"display": "none"}
    if tab == "patents-tab":
        return {"display": "none"}, {"display": "block"}, {"display": "none"}
    if tab == "combined-tab":
        return {"display": "none"}, {"display": "none"}, {"display": "block"}
    return {"display": "block"}, {"display": "none"}, {"display": "none"}


# ---------------------------------------------------------
# PAPERS MAP CALLBACK (paper-year-slider)
# ---------------------------------------------------------

@map.callback(
    Output("papers-map", "figure"),
    Input("papers-category-filter", "value"),
    Input("paper-year-slider", "value")
)
def update_papers_map(category_list, year_range):

    y_min, y_max = year_range

    df = PaperYears[(PaperYears["year"] >= y_min) & (PaperYears["year"] <= y_max)]

    if category_list:
        df = df[df["Category"].isin(category_list)]

    if df.empty:
        fig = px.scatter_map(pd.DataFrame({"lat": [], "lon": []}), lat="lat", lon="lon")
        fig.update_layout(map_style="carto-positron", mapbox_zoom=3)
        return fig

    agg = (
        df.groupby(["University", "Latitude", "Longitude"])["works_count"]
          .sum()
          .reset_index()
    )

    hover_dict = {
        univ: build_paper_hover(univ, df[df["University"] == univ])
        for univ in agg["University"].unique()
    }
    agg["hover"] = agg["University"].map(hover_dict)

    fig = px.density_map(
        agg, lat="Latitude", lon="Longitude", z="works_count",
        zoom=3, color_continuous_scale="Rainbow"
    )

    fig.add_scattermap(
        lat=agg["Latitude"], lon=agg["Longitude"],
        mode="markers", marker=dict(size=8, color="black"),
        text=agg["hover"]
    )

    fig.update_layout(map_style="carto-positron")
    return fig


# ---------------------------------------------------------
# PATENTS MAP CALLBACK (patent-year-slider)
# ---------------------------------------------------------

@map.callback(
    Output("patents-map", "figure"),
    Input("patents-category-filter", "value"),
    Input("patent-year-slider", "value")
)
def update_patents_map(category_list, year_range):

    y_min, y_max = year_range

    df = PatentYears[(PatentYears["year"] >= y_min) & (PatentYears["year"] <= y_max)]

    if category_list:
        df = df[df["Category"].isin(category_list)]

    if df.empty:
        fig = px.scatter_map(pd.DataFrame({"lat": [], "lon": []}), lat="lat", lon="lon")
        fig.update_layout(map_style="carto-positron", mapbox_zoom=3)
        return fig

    agg = (
        df.groupby(["University", "Latitude", "Longitude"])["patent_count"]
          .sum()
          .reset_index()
    )

    hover_dict = {
        univ: build_patent_hover(univ, df[df["University"] == univ])
        for univ in agg["University"].unique()
    }
    agg["hover"] = agg["University"].map(hover_dict)

    fig = px.density_map(
        agg, lat="Latitude", lon="Longitude", z="patent_count",
        zoom=3, color_continuous_scale="Rainbow"
    )

    fig.add_scattermap(
        lat=agg["Latitude"], lon=agg["Longitude"],
        mode="markers", marker=dict(size=8, color="black"),
        text=agg["hover"]
    )

    fig.update_layout(map_style="carto-positron")
    return fig


# ---------------------------------------------------------
# COMBINED MAP CALLBACK (combined-year-slider)
# ---------------------------------------------------------

@map.callback(
    Output("combined-map", "figure"),
    Input("combined-category-filter", "value"),
    Input("combined-year-slider", "value")
)
def update_combined_map(category_list, year_range):

    y_min, y_max = year_range

    df_p = PaperYears[(PaperYears["year"] >= y_min) & (PaperYears["year"] <= y_max)]
    df_t = PatentYears[(PatentYears["year"] >= y_min) & (PatentYears["year"] <= y_max)]

    if category_list:
        df_p = df_p[df_p["Category"].isin(category_list)]
        df_t = df_t[df_t["Category"].isin(category_list)]

    if df_p.empty and df_t.empty:
        fig = px.scatter_map(pd.DataFrame({"lat": [], "lon": []}), lat="lat", lon="lon")
        fig.update_layout(map_style="carto-positron", mapbox_zoom=3)
        return fig

    agg_p = (
        df_p.groupby(["University", "Latitude", "Longitude"])["works_count"]
            .sum()
            .reset_index()
    )

    agg_t = (
        df_t.groupby(["University", "Latitude", "Longitude"])["patent_count"]
            .sum()
            .reset_index()
    )

    merged = pd.merge(
        agg_p, agg_t,
        on=["University", "Latitude", "Longitude"],
        how="outer"
    ).fillna(0)

    merged["total"] = merged["works_count"] + merged["patent_count"]

    hover_dict = {}
    for univ in merged["University"].unique():
        p = int(merged.loc[merged["University"] == univ, "works_count"].iloc[0])
        t = int(merged.loc[merged["University"] == univ, "patent_count"].iloc[0])
        total = int(merged.loc[merged["University"] == univ, "total"].iloc[0])

        hover_dict[univ] = (
            f"<b>{univ.title()}</b><br>"
            f"Papers: {p}<br>"
            f"Patents: {t}<br>"
            f"Total: {total}"
        )

    merged["hover"] = merged["University"].map(hover_dict)

    fig = px.density_map(
        merged,
        lat="Latitude",
        lon="Longitude",
        z="total",
        zoom=3,
        color_continuous_scale="Rainbow"
    )

    fig.add_scattermap(
        lat=merged["Latitude"],
        lon=merged["Longitude"],
        mode="markers",
        marker=dict(size=8, color="black", opacity=0.85),
        text=merged["hover"]
    )

    fig.update_layout(map_style="carto-positron")
    return fig


# =====================================================================
# SELECTED UNIVERSITY TABLE — PAPERS MAP (patched with normalize_year)
# =====================================================================

@map.callback(
    Output("selected-university-title-papers", "children"),
    Output("university-table-papers", "data"),
    Input("papers-map", "clickData"),
    Input("papers-category-filter", "value"),
    Input("search-category", "value"),
    Input("carnegie-filter", "value"),
    Input("paper-year-slider", "value"),
)
def update_papers_table(click, papers_map_filter, categories, carnegie, year_range):

    if not click or "points" not in click:
        return "Select a university on the papers map", []

    hover_html = click["points"][0]["text"]
    clean_univ = strip_html(hover_html.split("<br>")[0])
    univ_key = normalize_univ(clean_univ)

    y_min, y_max = year_range

    # Normalize years BEFORE filtering
    df = PaperYears.copy()
    df["year"] = df["year"].apply(normalize_year)
    df = df.dropna(subset=["year"])

    # Carnegie filter
    if carnegie != "All":
        df_car = df[df["University"] == univ_key]
        if df_car.empty or df_car["Carnegie"].iloc[0] != carnegie:
            return f"Selected University: {clean_univ}", []

    # Apply filters
    df_univ = df[
        (df["University"] == univ_key) &
        (df["year"] >= y_min) &
        (df["year"] <= y_max)
    ]

    if papers_map_filter:
        df_univ = df_univ[df_univ["Category"].isin(papers_map_filter)]
    if categories:
        df_univ = df_univ[df_univ["Category"].isin(categories)]

    rows = []
    for _, row in df_univ.iterrows():
        key = (row["University"], row["Category"])
        works = Top5PapersDict.get(key, [])
        for w in works:
            rows.append({
                "title": w["title"],
                "type": "paper",
                "category": row["Category"],
                "Seed": w.get("seed", ""),
                "year": row["year"],  # unified + normalized
                "work_id": w.get("openalex_url", "")
            })

    if not rows:
        return f"Selected University: {clean_univ}", []

    df_out = pd.DataFrame(rows).drop_duplicates("work_id")
    df_out = df_out.sort_values("year", ascending=False)

    return f"Selected University: {clean_univ}", df_out.to_dict("records")


# =====================================================================
# SELECTED UNIVERSITY TABLE — PATENTS MAP (patched with normalize_year)
# =====================================================================

@map.callback(
    Output("selected-university-title-patents", "children"),
    Output("university-table-patents", "data"),
    Input("patents-map", "clickData"),
    Input("patents-category-filter", "value"),
    Input("search-category", "value"),
    Input("patent-year-slider", "value"),
    Input("carnegie-filter", "value"),
)
def update_patents_table(click, patents_map_filter, categories, year_range, carnegie):

    if not click or "points" not in click:
        return "Select a university on the patents map", []

    hover_html = click["points"][0]["text"]
    clean_univ = strip_html(hover_html.split("<br>")[0])
    univ_key = normalize_univ(clean_univ)

    y_min, y_max = year_range

    # Normalize years
    df = PatentYears.copy()
    df["year"] = df["year"].apply(normalize_year)
    df = df.dropna(subset=["year"])

    # Carnegie filter
    if carnegie != "All":
        df_car = df[df["University"] == univ_key]
        if df_car.empty or df_car["Carnegie"].iloc[0] != carnegie:
            return f"Selected University: {clean_univ}", []

    # Apply filters
    df_univ = df[
        (df["University"] == univ_key) &
        (df["year"] >= y_min) &
        (df["year"] <= y_max)
    ]

    if patents_map_filter:
        df_univ = df_univ[df_univ["Category"].isin(patents_map_filter)]
    if categories:
        df_univ = df_univ[df_univ["Category"].isin(categories)]

    rows = []
    for _, row in df_univ.iterrows():
        key = (row["University"], row["Category"])
        top5 = Top5PatentsDict.get(key, [])

        for patent in top5:
            rows.append({
                "title": patent.get("patent_title", ""),
                "type": "patent",
                "category": row["Category"],

                # Robust subcategory extraction
                "Seed": (
                    patent.get("seed")
                    or patent.get("Seed")
                    or patent.get("subcategory")
                    or patent.get("sub_category")
                    or patent.get("seeds")
                    or ""
                ),

                "year": row["year"],
                "work_id": patent["patent_id"]
            })

    if not rows:
        return f"Selected University: {clean_univ}", []

    df_out = pd.DataFrame(rows).drop_duplicates("work_id")
    df_out = df_out.sort_values("year", ascending=False)

    return f"Selected University: {clean_univ}", df_out.to_dict("records")


# =====================================================================
# SELECTED UNIVERSITY TABLE — COMBINED MAP (patched with normalize_year)
# =====================================================================

@map.callback(
    Output("selected-university-title-combined", "children"),
    Output("university-table-combined", "data"),
    Input("combined-map", "clickData"),
    Input("combined-category-filter", "value"),
    Input("combined-year-slider", "value"),
)
def update_combined_table(click, categories, year_range):

    if not click or "points" not in click:
        return "Select a university on the combined map", []

    hover_html = click["points"][0]["text"]
    clean_univ = strip_html(hover_html.split("<br>")[0])
    univ_key = normalize_univ(clean_univ)

    y_min, y_max = year_range

    # Normalize years for BOTH datasets
    df_p = PaperYears.copy()
    df_p["year"] = df_p["year"].apply(normalize_year)
    df_p = df_p.dropna(subset=["year"])

    df_t = PatentYears.copy()
    df_t["year"] = df_t["year"].apply(normalize_year)
    df_t = df_t.dropna(subset=["year"])

    # Apply filters
    df_p = df_p[
        (df_p["University"] == univ_key) &
        (df_p["year"] >= y_min) &
        (df_p["year"] <= y_max)
    ]
    df_t = df_t[
        (df_t["University"] == univ_key) &
        (df_t["year"] >= y_min) &
        (df_t["year"] <= y_max)
    ]

    if categories:
        df_p = df_p[df_p["Category"].isin(categories)]
        df_t = df_t[df_t["Category"].isin(categories)]

    rows = []

    # Papers
    for _, row in df_p.iterrows():
        key = (row["University"], row["Category"])
        works = Top5PapersDict.get(key, [])
        for w in works:
            rows.append({
                "title": w["title"],
                "type": "paper",
                "category": row["Category"],
                "Seed": w.get("seed", ""),
                "year": row["year"],
                "work_id": w.get("openalex_url", "")
            })

    # Patents
    for _, row in df_t.iterrows():
        key = (row["University"], row["Category"])
        top5 = Top5PatentsDict.get(key, [])
        for patent in top5:
            rows.append({
                "title": patent.get("patent_title", ""),
                "type": "patent",
                "category": row["Category"],
                "Seed": row["Seed"],
                "year": row["year"],
                "work_id": patent["patent_id"]
            })

    if not rows:
        return f"Selected University: {clean_univ}", []

    df_out = pd.DataFrame(rows).drop_duplicates("work_id")
    df_out = df_out.sort_values("year", ascending=False)

    return f"Selected University: {clean_univ}", df_out.to_dict("records")


# =====================================================================
# SEARCH — UPDATED FOR search-year-slider
# =====================================================================

@map.callback(
    Output("search-results-table", "data"),
    Input("search-category", "value"),
    Input("content", "value"),
    Input("carnegie-filter", "value"),
    Input("NUniversities", "value"),
    Input("search-university", "value"),
    Input("search-year-slider", "value"),
)
def update_search_results(categories, content_type,
                          carnegie_value, N, keyword,
                          year_range):

    y_min, y_max = year_range

    # ---------------------------------------------------------
    # NORMALIZE + UNIFY YEAR COLUMN BEFORE FILTERING
    # ---------------------------------------------------------
    df_p = PaperYears.copy()
    df_p["year"] = df_p["year"].apply(normalize_year)
    df_p = df_p.dropna(subset=["year"])

    df_t = PatentYears.copy()
    df_t["year"] = df_t["year"].apply(normalize_year)
    df_t = df_t.dropna(subset=["year"])

    # ---------------------------------------------------------
    # YEAR FILTER
    # ---------------------------------------------------------
    df_p = df_p[(df_p["year"] >= y_min) & (df_p["year"] <= y_max)]
    df_t = df_t[(df_t["year"] >= y_min) & (df_t["year"] <= y_max)]

    # ---------------------------------------------------------
    # CATEGORY FILTER
    # ---------------------------------------------------------
    if categories:
        df_p = df_p[df_p["Category"].isin(categories)]
        df_t = df_t[df_t["Category"].isin(categories)]

    # ---------------------------------------------------------
    # CARNEGIE FILTER
    # ---------------------------------------------------------
    if carnegie_value != "All":
        df_p = df_p[df_p["Carnegie"] == carnegie_value]
        df_t = df_t[df_t["Carnegie"] == carnegie_value]

    # ---------------------------------------------------------
    # CONTENT TYPE FILTER
    # ---------------------------------------------------------
    if content_type == "papers":
        df_t = df_t.iloc[0:0]
    elif content_type == "patents":
        df_p = df_p.iloc[0:0]

    # ---------------------------------------------------------
    # MERGE WITH UNIFIED YEAR COLUMN
    # ---------------------------------------------------------
    merged = pd.merge(
        df_p, df_t,
        on=["University", "Category", "Carnegie", "Latitude", "Longitude", "State", "year"],
        how="outer",
        suffixes=("_papers", "_patents")
    )

    merged["works_count"] = merged["works_count"].fillna(0).astype(int)
    merged["patent_count"] = merged["patent_count"].fillna(0).astype(int)

    rows = []

    # ---------------------------------------------------------
    # BUILD ROWS
    # ---------------------------------------------------------
    for _, row in merged.iterrows():

        univ = row["University"]
        category = row["Category"]
        state = row["State"]
        year_val = row["year"]  # unified year

        # -------------------------
        # PAPERS
        # -------------------------
        if row["works_count"] > 0 and content_type in ("papers", "both"):
            key = (univ, category)
            if key in Top5PapersDict:
                for w in Top5PapersDict[key]:
                    rows.append({
                        "University": univ.title(),
                        "State": state,
                        "Carnegie": row["Carnegie"],
                        "Category": category,
                        "Seed": w.get("seed", ""),
                        "works_count": row["works_count"],
                        "us_patents_count": row["patent_count"],
                        "type": "paper",
                        "title": w["title"],
                        "year": year_val,  # unified year
                        "cited_by_count": w.get("cited_by_count", ""),
                        "ID": w.get("openalex_url", "")
                    })

        # -------------------------
        # PATENTS
        # -------------------------
        if row["patent_count"] > 0 and content_type in ("patents", "both"):

            key = (univ, category)
            top5 = Top5PatentsDict.get(key, [])

            for patent in top5:
                rows.append({
                    "University": univ.title(),
                    "State": state,
                    "Carnegie": row["Carnegie"],
                    "Category": category,
                    "Seed": patent.get("seed", row.get("Seed_patents", "")),
                    "works_count": row["works_count"],
                    "us_patents_count": row["patent_count"],
                    "type": "patent",
                    "title": patent.get("patent_title", ""),
                    "year": year_val,  # unified year
                    "cited_by_count": "",
                    "ID": patent["patent_id"]
                })

    if not rows:
        return []

    df_rows = pd.DataFrame(rows).drop_duplicates(subset=["ID"])

    # ---------------------------------------------------------
    # KEYWORD SEARCH
    # ---------------------------------------------------------
    if keyword:
        kw = keyword.lower()
        df_rows = df_rows[
            df_rows["University"].str.lower().str.contains(kw, na=False)
            | df_rows["Category"].str.lower().str.contains(kw, na=False)
            | df_rows["Seed"].str.lower().str.contains(kw, na=False)
            | df_rows["title"].str.lower().str.contains(kw, na=False)
            | df_rows["State"].str.lower().str.contains(kw, na=False)
            | df_rows["ID"].str.lower().str.contains(kw, na=False)
        ]

    # ---------------------------------------------------------
    # FINAL SORT
    # ---------------------------------------------------------
    df_rows = df_rows.sort_values("year", ascending=False)

    return df_rows.head(N).to_dict("records")

# =====================================================================
# UNIVERSITY OUTPUT OVER TIME — PAPERS / PATENTS / BOTH
# =====================================================================

@map.callback(
    Output("patent-university-graphs", "children"),
    Input("uot-content-type", "value"),
    Input("patent-regionselect", "value"),
    Input("patent-NUniversities", "value"),
    Input("patent-USearch", "value")
)
def update_output_over_time(content_type, region, num_unis, usearch):

    if region is None:
        region = "All Regions"

    # ---------------------------------------------------------
    # SELECT DATASET(S)
    # ---------------------------------------------------------
    if content_type == "papers":
        df = PaperYears.copy()
        count_col = "works_count"

    elif content_type == "patents":
        df = PatentYears.copy()
        count_col = "patent_count"

    else:  # BOTH
        df_p = PaperYears.copy()
        df_t = PatentYears.copy()

        df = pd.merge(
            df_p,
            df_t,
            on=["University", "Category", "year", "State"],
            how="outer",
            suffixes=("_p", "_t")
        ).fillna(0)

        df["count"] = df["works_count"] + df["patent_count"]
        count_col = "count"

    # ---------------------------------------------------------
    # KEYWORD SEARCH
    # ---------------------------------------------------------
    if usearch:
        df = df[
            df["University"].str.contains(usearch, case=False, na=False)
            | df["State"].str.contains(usearch, case=False, na=False)
        ]

    # ---------------------------------------------------------
    # REGION FILTER
    # ---------------------------------------------------------
    region_unis = df[df["State"].isin(regions[region])]["University"].unique()
    df = df[df["University"].isin(region_unis)]

    if df.empty:
        return [html.Div("No data available for selected filters.")]

    # ---------------------------------------------------------
    # AGGREGATE
    # ---------------------------------------------------------
    agg = (
        df.groupby(["University", "Category", "year"])[count_col]
          .sum()
          .reset_index(name="count")
    )

    # Normalize year values
    agg["year"] = agg["year"].apply(normalize_year)
    agg = agg.dropna(subset=["year"])

    if agg.empty:
        return [html.Div("No valid year data available.")]

    latest_year = agg["year"].max()

    # Ranking by latest year
    year_totals = (
        agg[agg["year"] == latest_year]
        .groupby("University")["count"]
        .sum()
        .sort_values(ascending=False)
    )

    ranking = year_totals.head(num_unis).index.tolist()

    df_top = agg[agg["University"].isin(ranking)]

    # ---------------------------------------------------------
    # BUILD GRAPHS
    # ---------------------------------------------------------
    graphs = []
    for uni in ranking:
        df_uni = df_top[df_top["University"] == uni].sort_values("year")

        fig = px.line(
            df_uni,
            x="year",
            y="count",
            color="Category",
            markers=True,
            title=uni.title(),
        )

        fig.update_layout(
            height=300,
            margin=dict(l=20, r=20, t=40, b=20)
        )

        graphs.append(
            html.Div(
                [dcc.Graph(figure=fig)],
                style={"marginBottom": "40px"}
            )
        )

    return graphs
    
# ---------------------------------------------------------
# RUN APP
# ---------------------------------------------------------

if __name__ == "__main__":
    map.run(debug=True, port=8037)
